# Flexible Search via GPU

This notebook has code to perform searches over $\prod_{i} [m_i,n_i]$. It uses the improved dimension computation algorithm.

We also allow safe interruption of the search if it turnsout to take far too long.

In [174]:
# %load ../dim_backprop_gpu_only.py
"""Exact GPU computation of polynomial-network neurovariety dimensions.

This module has no SageMath or NumPy backend.  It requires CuPy and a
CUDA-capable NVIDIA GPU.

The public function keeps the original calling convention::

    compute_dimension(network_widths, network_exponent)

The returned tuple is

    (sizes, exponent, ambient_dim, expected_dim, dimension, defect).

The computation is exact over the finite fields listed in ``DEFAULT_PRIMES``.
All samples and all output-coordinate pullbacks are differentiated in one
batched GPU computation.  The final Jacobian rank is computed modulo each
prime by GPU-parallel row elimination.
"""

from __future__ import annotations

from math import comb
from typing import Iterator, Sequence

try:
    import cupy as cp
except ImportError as exc:
    raise RuntimeError(
        "This GPU-only module requires CuPy. Install the CuPy package that "
        "matches your CUDA version, for example `pip install cupy-cuda12x`."
    ) from exc


DEFAULT_PRIMES = (100003, 100153)
_INT64_MAX = (1 << 63) - 1


def _require_cuda() -> None:
    """Raise a clear error unless a usable CUDA device is visible."""
    try:
        device_count = int(cp.cuda.runtime.getDeviceCount())
    except cp.cuda.runtime.CUDARuntimeError as exc:
        raise RuntimeError(
            "CuPy is installed, but CUDA could not be initialized. Check the "
            "NVIDIA driver, CUDA/CuPy compatibility, and notebook kernel."
        ) from exc

    if device_count < 1:
        raise RuntimeError("No CUDA-capable GPU is visible to CuPy.")


def gpu_information() -> dict[str, object]:
    """Return basic information about the CUDA device used by this module."""
    _require_cuda()
    device_id = int(cp.cuda.runtime.getDevice())
    properties = cp.cuda.runtime.getDeviceProperties(device_id)
    name = properties["name"]
    if isinstance(name, bytes):
        name = name.decode("utf-8", errors="replace")
    return {
        "device_id": device_id,
        "name": name,
        "device_count": int(cp.cuda.runtime.getDeviceCount()),
        "cupy_version": cp.__version__,
        "cuda_runtime_version": int(cp.cuda.runtime.runtimeGetVersion()),
        "driver_version": int(cp.cuda.runtime.driverGetVersion()),
    }


def _is_prime(value: int) -> bool:
    """Deterministic Miller--Rabin test for unsigned 64-bit integers."""
    if value < 2:
        return False

    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    if value in small_primes:
        return True
    if any(value % prime == 0 for prime in small_primes):
        return False

    odd_part = value - 1
    power_of_two = 0
    while odd_part % 2 == 0:
        power_of_two += 1
        odd_part //= 2

    # Deterministic for every n < 2^64.
    for base in (2, 325, 9375, 28178, 450775, 9780504, 1795265022):
        if base % value == 0:
            continue
        witness = pow(base, odd_part, value)
        if witness in (1, value - 1):
            continue
        for _ in range(power_of_two - 1):
            witness = (witness * witness) % value
            if witness == value - 1:
                break
        else:
            return False

    return True


def _weak_compositions(total: int, length: int) -> Iterator[tuple[int, ...]]:
    """Yield nonnegative ``length``-tuples whose entries sum to ``total``."""
    if length == 0:
        if total == 0:
            yield ()
        return
    if length == 1:
        yield (total,)
        return

    for first in range(total + 1):
        for rest in _weak_compositions(total - first, length - 1):
            yield (first,) + rest


def _unisolvent_samples(input_dim: int, degree: int, prime: int):
    r"""Construct the homogeneous interpolation points directly on the GPU.

    On the chart ``x_0 = 1``, homogeneous degree-``degree`` forms become
    polynomials of total degree at most ``degree`` in ``input_dim - 1``
    variables.  The integer simplex is an unisolvent evaluation set when
    ``prime > degree``.
    """
    if input_dim < 1:
        raise ValueError("the input layer must have positive width")
    if degree < 0:
        raise ValueError("degree must be nonnegative")
    if prime <= degree:
        raise ValueError(
            f"prime {prime} must exceed polynomial degree {degree}"
        )

    expected = comb(degree + input_dim - 1, input_dim - 1)
    if input_dim == 1:
        return cp.ones((1, 1), dtype=cp.int64)

    points: list[tuple[int, ...]] = []
    for total in range(degree + 1):
        for alpha in _weak_compositions(total, input_dim - 1):
            points.append((1,) + alpha)

    if len(points) != expected:
        raise RuntimeError(
            f"internal sample-count error: got {len(points)}, expected {expected}"
        )

    return cp.asarray(points, dtype=cp.int64) % prime


def _mod_pow(base, exponent: int, prime: int):
    """Elementwise modular exponentiation on the GPU."""
    if exponent < 0:
        raise ValueError("exponent must be nonnegative")

    result = cp.ones_like(base, dtype=cp.int64)
    if exponent == 0:
        return result

    power = base.astype(cp.int64, copy=False) % prime
    remaining = int(exponent)
    while remaining:
        if remaining & 1:
            result = (result * power) % prime
        remaining >>= 1
        if remaining:
            power = (power * power) % prime

    return result


def _check_dot_product_safety(widths: Sequence[int], prime: int) -> None:
    """Prevent signed int64 overflow before a matrix product is reduced."""
    largest_inner_dimension = max(int(width) for width in widths)
    worst_case = largest_inner_dimension * (prime - 1) ** 2
    if worst_case > _INT64_MAX:
        raise OverflowError(
            "An int64 GPU matrix product may overflow before reduction modulo "
            "the prime. Use a smaller prime or narrower layers."
        )


def _random_weights(
    widths: Sequence[int], prime: int, seed: int
) -> list[cp.ndarray]:
    """Generate all network weight matrices directly in GPU memory."""
    rng = cp.random.RandomState(seed)
    return [
        rng.randint(
            0,
            prime,
            size=(out_width, in_width),
            dtype=cp.int64,
        )
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ]


def _parameter_offsets(
    weights: Sequence[cp.ndarray],
) -> tuple[list[int], int]:
    offsets: list[int] = []
    total = 0
    for weight in weights:
        offsets.append(total)
        total += int(weight.shape[0] * weight.shape[1])
    return offsets, total


def _batched_weight_jacobian(
    weights: Sequence[cp.ndarray],
    samples: cp.ndarray,
    exponent: int,
    prime: int,
) -> cp.ndarray:
    """Evaluate every output/weight derivative at every sample on the GPU.

    The result has shape ``(number_of_samples, output_width, num_parameters)``.
    Weight matrices are flattened layer by layer in row-major order, matching
    the ordering used by Sage's matrix ``list()`` method in the old code.
    """
    if exponent < 1:
        raise ValueError("network exponent must be at least 1")
    if not weights:
        raise ValueError("the network must contain at least one weight layer")

    activations = [samples]
    preactivations: list[cp.ndarray] = []
    activation = samples

    # Hidden layers use z -> z^exponent; the final layer is linear.
    for weight in weights[:-1]:
        preactivation = cp.matmul(activation, weight.T) % prime
        preactivations.append(preactivation)
        activation = _mod_pow(preactivation, exponent, prime)
        activations.append(activation)

    batch_size = int(samples.shape[0])
    output_width = int(weights[-1].shape[0])
    offsets, num_parameters = _parameter_offsets(weights)

    jacobian = cp.zeros(
        (batch_size, output_width, num_parameters), dtype=cp.int64
    )

    # Final linear layer.
    final_input = activations[-1]
    final_offset = offsets[-1]
    final_input_width = int(weights[-1].shape[1])
    for output_index in range(output_width):
        start = final_offset + output_index * final_input_width
        stop = start + final_input_width
        jacobian[:, output_index, start:stop] = final_input

    if len(weights) == 1:
        return jacobian

    # Derivatives of all output coordinates with respect to the last hidden
    # preactivation: shape (sample, output, hidden neuron).
    delta = cp.broadcast_to(
        weights[-1][None, :, :],
        (batch_size, output_width, int(weights[-1].shape[1])),
    ).copy()
    derivative = (
        (exponent % prime)
        * _mod_pow(preactivations[-1], exponent - 1, prime)
    ) % prime
    delta = (delta * derivative[:, None, :]) % prime

    # Hidden layers from last to first.
    for layer_index in range(len(weights) - 2, -1, -1):
        weight = weights[layer_index]
        layer_input = activations[layer_index]

        gradient = (
            delta[:, :, :, None] * layer_input[:, None, None, :]
        ) % prime

        start = offsets[layer_index]
        stop = start + int(weight.shape[0] * weight.shape[1])
        jacobian[:, :, start:stop] = gradient.reshape(
            batch_size, output_width, stop - start
        )

        if layer_index > 0:
            delta = cp.matmul(delta, weight) % prime
            derivative = (
                (exponent % prime)
                * _mod_pow(
                    preactivations[layer_index - 1], exponent - 1, prime
                )
            ) % prime
            delta = (delta * derivative[:, None, :]) % prime

    return jacobian


def _rank_mod_prime_gpu(
    matrix: cp.ndarray,
    prime: int,
    workspace_bytes: int = 512 * 1024**2,
) -> int:
    """Compute exact matrix rank over GF(prime) using GPU row operations."""
    if matrix.ndim != 2:
        raise ValueError("rank input must be a matrix")
    if workspace_bytes <= 0:
        raise ValueError("workspace_bytes must be positive")

    reduced = matrix.astype(cp.int64, copy=True) % prime

    # Eliminate along the smaller dimension.
    if reduced.shape[1] > reduced.shape[0]:
        reduced = reduced.T.copy()

    nrows, ncols = map(int, reduced.shape)
    pivot_row = 0

    for column in range(ncols):
        if pivot_row == nrows:
            break

        nonzero = reduced[pivot_row:, column] != 0
        if not bool(cp.any(nonzero).item()):
            continue

        pivot = pivot_row + int(cp.argmax(nonzero).item())
        if pivot != pivot_row:
            temporary = reduced[pivot_row, :].copy()
            reduced[pivot_row, :] = reduced[pivot, :]
            reduced[pivot, :] = temporary

        pivot_value = int(reduced[pivot_row, column].item())
        inverse = pow(pivot_value, prime - 2, prime)
        reduced[pivot_row, column:] = (
            reduced[pivot_row, column:] * inverse
        ) % prime

        # The row updates are parallel CUDA kernels. Chunking bounds temporary
        # memory usage for large Jacobians.
        remaining_columns = ncols - column
        bytes_per_row = max(1, 3 * remaining_columns * 8)
        rows_per_chunk = max(1, workspace_bytes // bytes_per_row)

        start = pivot_row + 1
        while start < nrows:
            stop = min(nrows, start + rows_per_chunk)
            factors = reduced[start:stop, column].copy()
            reduced[start:stop, column:] = (
                reduced[start:stop, column:]
                - factors[:, None] * reduced[pivot_row, column:][None, :]
            ) % prime
            start = stop

        pivot_row += 1

    return pivot_row


def compute_dimension(
    network_widths: Sequence[int],
    network_exponent: int,
    *,
    primes: Sequence[int] = DEFAULT_PRIMES,
    seed: int = 20260630,
    rank_workspace_bytes: int = 512 * 1024**2,
    verbose: bool = False,
):
    """Compute the neurovariety dimension entirely with the CUDA backend.

    Parameters
    ----------
    network_widths:
        Layer widths ``[d0, d1, ..., dL]``.
    network_exponent:
        Common hidden-layer activation exponent.
    primes:
        Prime moduli used to cross-check the generic rank.
    seed:
        Base random seed for the network weights.
    rank_workspace_bytes:
        Approximate upper bound for temporary elimination workspace.
    verbose:
        Print GPU and rank information.

    Returns
    -------
    tuple
        ``(sizes, exponent, ambient_dim, expected_dim, dimension, defect)``.
    """
    _require_cuda()

    widths = tuple(int(width) for width in network_widths)
    exponent = int(network_exponent)

    if len(widths) < 2:
        raise ValueError("network_widths must contain input and output widths")
    if any(width <= 0 for width in widths):
        raise ValueError("all network widths must be positive")
    if exponent < 1:
        raise ValueError("network_exponent must be at least 1")
    if not primes:
        raise ValueError("at least one prime is required")

    degree = exponent ** (len(widths) - 2)
    ambient_per_output = comb(degree + widths[0] - 1, widths[0] - 1)
    ambient_dim = ambient_per_output * widths[-1]
    num_parameters = sum(
        in_width * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    )

    dimensions: list[int] = []

    if verbose:
        info = gpu_information()
        print(
            f"GPU {info['device_id']}: {info['name']} | "
            f"CuPy {info['cupy_version']} | "
            f"CUDA runtime {info['cuda_runtime_version']}"
        )

    for prime_index, prime_value in enumerate(primes):
        prime = int(prime_value)
        if not _is_prime(prime):
            raise ValueError(f"modulus {prime} is not prime")
        _check_dot_product_safety(widths, prime)

        samples = _unisolvent_samples(widths[0], degree, prime)
        weights = _random_weights(
            widths,
            prime,
            seed + 1_000_003 * prime_index + prime,
        )

        jacobian = _batched_weight_jacobian(
            weights,
            samples,
            exponent,
            prime,
        )
        rank_matrix = jacobian.reshape(
            ambient_per_output * widths[-1], num_parameters
        )
        dimension = _rank_mod_prime_gpu(
            rank_matrix,
            prime,
            workspace_bytes=rank_workspace_bytes,
        )
        dimensions.append(dimension)

        # Ensure kernels for this prime have completed before reporting and
        # releasing memory.
        cp.cuda.Stream.null.synchronize()

        if verbose:
            print(
                f"prime={prime}, samples={ambient_per_output}, "
                f"rank_matrix={tuple(rank_matrix.shape)}, rank={dimension}"
            )

        del rank_matrix, jacobian, weights, samples
        cp.get_default_memory_pool().free_all_blocks()

    if not all(dimension == dimensions[0] for dimension in dimensions):
        raise ValueError(
            "different dimensions over finite fields: " + str(dimensions)
        )

    naive_bound = sum(
        (in_width - 1) * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ) + widths[-1]
    expected_dim = min(ambient_dim, naive_bound)
    dimension = dimensions[0]

    return (
        list(widths),
        exponent,
        ambient_dim,
        expected_dim,
        dimension,
        expected_dim - dimension,
    )


if __name__ == "__main__":
    print(compute_dimension([2, 3, 1], 2, verbose=True))


GPU 0: NVIDIA GeForce RTX 4070 SUPER | CuPy 14.1.1 | CUDA runtime 12090
prime=100003, samples=3, rank_matrix=(3, 9), rank=3
prime=100153, samples=3, rank_matrix=(3, 9), rank=3
([2, 3, 1], 2, 3, 3, 3, 0)


# Search Algorithm

In [175]:
import os
import ast
import math
import itertools
import random as py_random
import pandas as pd
from tqdm import tqdm

In [176]:
# Helper Functions

def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Calculates the total number of weights and biases in the network."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))

def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Returns True if every element in t1 is <= the corresponding element in t2."""
    if len(t1) != len(t2):
        return False
    return all(a <= b for a, b in zip(t1, t2))

def evaluate_single_architecture(hidden_tuple: tuple, h: int, d_0: int, d_h: int, exponent: int):
    """Worker function to evaluate a single architecture and format the result."""
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)
    
    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = (dim == amb)
        
        status = "FULL " if is_full else "SHORT"
        print(f"  [{status}] {arch_str} -> Rank: {dim}/{amb} (Params: {params})")
        
        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": is_full,
            "is_minimal": False 
        }
    except Exception as e:
        print(f"  [ERROR] {arch_str} failed: {e}")
        return None

In [177]:
# Core Search & Pruning Algorithm

def parameter_boundary_search(
    h_values: list, max_width=6, min_width=1, exponent=2, 
    d_0=2, d_h=1, csv_filename="architecture_search_log.csv", 
    user_guesses=None, layer_bounds=None
):

    # Load or initialize Database
    if os.path.exists(csv_filename):
        print(f"Loading existing database from '{csv_filename}'...")
        df = pd.read_csv(csv_filename)
    else:
        print("No existing database found. Starting fresh...")
        df = pd.DataFrame(columns=[
            "h", "exponent", "architecture", "num_parameters", 
            "dimension_computed", "ambient_dimension", "is_full_dimension", "is_minimal"
        ])

    evaluated_architectures = set(df["architecture"].tolist())
    new_records = []

    for h in h_values:
        print(f"\n--- RANDOM SEARCH: h={h}, exponent={exponent} ---")
        
        num_hidden = h - 1
        degree = exponent ** num_hidden
        ambient_dim = math.comb(degree + d_0 - 1, d_0 - 1) * d_h
        print(f"  Target Ambient Dimension: {ambient_dim}")

        # Extract known boundaries
        known_minimal = set()
        known_short = set()
        
        if not df.empty:
            subset_df = df[(df["h"] == h) & (df["exponent"] == exponent)]
            
            min_df = subset_df[subset_df["is_minimal"] == True]
            known_minimal.update(tuple(ast.literal_eval(a)[1:-1]) for a in min_df["architecture"])
            
            short_df = subset_df[subset_df["is_full_dimension"] == False]
            known_short.update(tuple(ast.literal_eval(a)[1:-1]) for a in short_df["architecture"])

        # 1. Evaluate User Guesses
        if user_guesses and h in user_guesses:
            for guess in user_guesses[h]:
                arch_str = str([d_0] + list(guess) + [d_h])
                if arch_str in evaluated_architectures:
                    continue
                
                print(f"  Evaluating Guess: {guess}...")
                res = evaluate_single_architecture(guess, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                if res["is_full_dimension"]:
                    known_minimal.add(guess)
                else:
                    known_short.add(guess)

# 2. Build and Filter Search Pool
        print("  Generating candidate pool...")
        
        # Builds the Search Pool but imposing bounds on the individual layers

        if layer_bounds and h in layer_bounds:
            bounds = layer_bounds[h]
            if len(bounds) != num_hidden:
                print(f"  [WARNING] layer_bounds for h={h} has length {len(bounds)}, but expected {num_hidden} hidden layers. Skipping this depth...")
                continue
            
            print(f"  Using custom per-layer bounds: {bounds}")
            ranges = [range(max(1, b_min), b_max + 1) for b_min, b_max in bounds]
            all_possible = itertools.product(*ranges)
            
            # calculate total items for the progress bar
            total_combinations = math.prod(len(r) for r in ranges)
            
        else:
            print(f"  Using global bounds: min_width={min_width}, max_width={max_width}")
            width_range = range(max(1, min_width), max_width + 1)
            all_possible = itertools.product(width_range, repeat=num_hidden)
            
            # calculate total items for the progress bar
            total_combinations = len(width_range) ** num_hidden
        
        candidate_pool = []
        rejected_count = 0
        
        # wrap in tqdm. useful for long run times.
        for hidden in tqdm(all_possible, total=total_combinations, desc="  Filtering", leave=False, dynamic_ncols=True):
            arch_str = str([d_0] + list(hidden) + [d_h])
            if arch_str in evaluated_architectures:
                continue
                
            params = calculate_parameter_count(hidden, d_0, d_h)
            
            # pruning 
            if (params < ambient_dim or 
                any(is_less_or_equal(m, hidden) for m in known_minimal) or 
                any(is_less_or_equal(hidden, s) for s in known_short)):
                rejected_count += 1
                continue
                
            candidate_pool.append(hidden)
            
        print(f"  Pruned {rejected_count} impossible/redundant architectures.")
        print(f"  Starting sequential evaluation on {len(candidate_pool)} viable candidates.")

        # shuffle queue 
        py_random.shuffle(candidate_pool)

        try:
            while candidate_pool:
                target = candidate_pool.pop(0)
                
                res = evaluate_single_architecture(target, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                before_len = len(candidate_pool)
                if res["is_full_dimension"]:
                    known_minimal.add(target)
                    # remove all supersets from the queue
                    candidate_pool = [c for c in candidate_pool if not is_less_or_equal(target, c)]
                    pruned = before_len - len(candidate_pool)
                    if pruned > 0: print(f"    [Upward Pruned] {pruned} supersets removed from queue.")
                else:
                    known_short.add(target)
                    # remove all subsets from the queue
                    candidate_pool = [c for c in candidate_pool if not is_less_or_equal(c, target)]
                    pruned = before_len - len(candidate_pool)
                    if pruned > 0: print(f"    [Downward Pruned] {pruned} subsets removed from queue.")

        except KeyboardInterrupt: # incase one wants to halt a long run early and still save the results
            print("\n[Interrupt] User halted the loop early. Saving progress and continuing execution...")

    # 4. Save and determine minimality based on results in the .csv
    # This does not guarantee the examples are truly minimality -- need to do actual checks elsewhere.
    if new_records:
        new_df = pd.DataFrame(new_records).dropna(axis=1, how='all')
        if not df.empty:
            df = pd.concat([df, new_df], ignore_index=True)
        else:
            df = new_df
        print(f"\nAdded {len(new_records)} new architectures to the database.")

    print("Re-evaluating minimal filling properties (grouped by depth h and exponent)...")
    if not df.empty:
        df["is_minimal"] = False 
        
        # calculate if minimal 
        # this does not actual check minimality fully -- just minimality base on search so far
        # see verify_minimal.ipynb

        for (h_val, exp_val), group in df[df["is_full_dimension"] == True].groupby(["h", "exponent"]):
            full_archs = [ast.literal_eval(arch) for arch in group["architecture"]]
            
            for idx, row in group.iterrows():
                parsed_config = ast.literal_eval(row["architecture"])
                # it is minimal if no other full architecture is strictly smaller than it
                is_min = not any(
                    other != parsed_config and is_less_or_equal(other, parsed_config) 
                    for other in full_archs
                )
                df.at[idx, "is_minimal"] = is_min

        df.to_csv(csv_filename, index=False)
        print(f"Database successfully saved to '{csv_filename}'.\n")
    return df

# Search Algorithm 2

In [178]:
# from __future__ import annotations

# import ast
# import heapq
# import math
# import os
# import random as py_random
# from pathlib import Path
# from typing import Iterable, Sequence

# import pandas as pd


# # =============================================================================
# # Helpers
# # =============================================================================

# def _is_less_or_equal(a: Sequence[int], b: Sequence[int]) -> bool:
#     """
#     Coordinatewise comparison.

#     If the notebook already defines `is_less_or_equal`, that implementation is
#     used. Otherwise, this function supplies the usual coordinatewise order.
#     """
#     try:
#         return bool(is_less_or_equal(a, b))
#     except NameError:
#         return len(a) == len(b) and all(x <= y for x, y in zip(a, b))


# def _as_bool(value: object) -> bool:
#     """Parse booleans stored by pandas or in a CSV."""
#     if pd.isna(value):
#         return False
#     if isinstance(value, str):
#         return value.strip().lower() in {"true", "1", "yes"}
#     return bool(value)


# def _parse_architecture(value: object) -> tuple[int, ...]:
#     """Parse an architecture stored either as a list or as a CSV string."""
#     if isinstance(value, str):
#         value = ast.literal_eval(value)
#     return tuple(int(x) for x in value)


# def _parameter_count(
#     hidden: Sequence[int],
#     d_0: int,
#     d_h: int,
# ) -> int:
#     """
#     Use the project's existing parameter-count function when available.

#     The fallback is the standard parameter-dimension bound
#         sum_i d_{i-1}d_i - sum_hidden d_i.
#     """
#     try:
#         return int(calculate_parameter_count(hidden, d_0, d_h))
#     except NameError:
#         architecture = (d_0, *hidden, d_h)
#         edge_parameters = sum(
#             architecture[i] * architecture[i + 1]
#             for i in range(len(architecture) - 1)
#         )
#         hidden_scalings = sum(hidden)
#         return edge_parameters - hidden_scalings


# def _insert_minimal(
#     antichain: set[tuple[int, ...]],
#     point: tuple[int, ...],
# ) -> set[tuple[int, ...]]:
#     """Insert `point` into an antichain of coordinatewise minimal elements."""
#     if any(_is_less_or_equal(other, point) for other in antichain):
#         return antichain

#     return {
#         other
#         for other in antichain
#         if not _is_less_or_equal(point, other)
#     } | {point}


# def _insert_maximal(
#     antichain: set[tuple[int, ...]],
#     point: tuple[int, ...],
# ) -> set[tuple[int, ...]]:
#     """Insert `point` into an antichain of coordinatewise maximal elements."""
#     if any(_is_less_or_equal(point, other) for other in antichain):
#         return antichain

#     return {
#         other
#         for other in antichain
#         if not _is_less_or_equal(other, point)
#     } | {point}


# def _minimal_antichain(
#     points: Iterable[tuple[int, ...]],
# ) -> set[tuple[int, ...]]:
#     result: set[tuple[int, ...]] = set()
#     for point in sorted(set(points), key=lambda x: (sum(x), x)):
#         result = _insert_minimal(result, point)
#     return result


# def _maximal_antichain(
#     points: Iterable[tuple[int, ...]],
# ) -> set[tuple[int, ...]]:
#     result: set[tuple[int, ...]] = set()
#     for point in sorted(set(points), key=lambda x: (-sum(x), x)):
#         result = _insert_maximal(result, point)
#     return result


# def _save_csv(df: pd.DataFrame, csv_path: Path) -> None:
#     """
#     Save directly to the requested CSV, as in the original implementation.

#     No .pkl checkpoint and no companion state file are created.
#     """
#     csv_path.parent.mkdir(parents=True, exist_ok=True)
#     df.to_csv(csv_path, index=False)


# def _append_result(
#     df: pd.DataFrame,
#     result: dict,
#     *,
#     hidden: tuple[int, ...],
#     h: int,
#     exponent: int,
#     d_0: int,
#     d_h: int,
#     ambient_dim: int,
# ) -> pd.DataFrame:
#     """Append one evaluator result while preserving the original CSV schema."""
#     row = dict(result)

#     row.setdefault("h", h)
#     row.setdefault("exponent", exponent)
#     row.setdefault("architecture", str([d_0, *hidden, d_h]))
#     row.setdefault("num_parameters", _parameter_count(hidden, d_0, d_h))
#     row.setdefault("ambient_dimension", ambient_dim)
#     row.setdefault("is_minimal", False)

#     architecture = _parse_architecture(row["architecture"])
#     row["architecture"] = str(list(architecture))

#     df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

#     return df.drop_duplicates(
#         subset=["h", "exponent", "architecture"],
#         keep="last",
#     )


# def _recompute_minimality(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Recompute minimality among all filling architectures currently in the CSV.

#     This has the same meaning as in the original code: it is minimality relative
#     to the filling architectures found so far.
#     """
#     if df.empty:
#         return df

#     df = df.copy()
#     df["is_minimal"] = False

#     full_rows = df[df["is_full_dimension"].map(_as_bool)]

#     for (_, _), group in full_rows.groupby(["h", "exponent"]):
#         architectures = {
#             index: _parse_architecture(row["architecture"])
#             for index, row in group.iterrows()
#         }

#         for index, architecture in architectures.items():
#             df.at[index, "is_minimal"] = not any(
#                 other_index != index
#                 and other_architecture != architecture
#                 and _is_less_or_equal(other_architecture, architecture)
#                 for other_index, other_architecture in architectures.items()
#             )

#     return df


# # =============================================================================
# # Core Search & Pruning Algorithm
# # =============================================================================

# def parameter_boundary_search(
#     h_values: list,
#     max_width=6,
#     min_width=1,
#     exponent=2,
#     d_0=2,
#     d_h=1,
#     csv_filename="architecture_search_log.csv",
#     user_guesses=None,
#     layer_bounds=None,
# ):
#     """
#     Drop-in replacement for the original randomized boundary search.

#     The function signature and the user's execution block are unchanged.

#     Main differences from the original implementation
#     -------------------------------------------------
#     1. The full Cartesian product is never constructed.
#     2. No `candidate_pool` list is materialized.
#     3. Candidates are generated lazily from the current nonfilling frontier.
#     4. The exposed frontier is processed in random order.
#     5. Existing CSV results are loaded before the search and used for:
#          - exact-result reuse;
#          - upward pruning above known filling architectures;
#          - downward pruning below known nonfilling architectures.
#     6. Every completed evaluator call is written immediately to the same CSV.
#     7. On KeyboardInterrupt, the CSV is updated and returned. No .pkl file is
#        created.

#     Important
#     ---------
#     The randomization is among the currently exposed frontier points. This
#     retains the practical benefit of random search without paying the cost of
#     constructing and shuffling the full candidate pool.
#     """
#     required_columns = [
#         "h",
#         "exponent",
#         "architecture",
#         "num_parameters",
#         "dimension_computed",
#         "ambient_dimension",
#         "is_full_dimension",
#         "is_minimal",
#     ]

#     csv_path = Path(csv_filename)

#     # -------------------------------------------------------------------------
#     # Load or initialize the original database.
#     # -------------------------------------------------------------------------
#     if csv_path.exists():
#         print(
#             f"Loading existing database from '{csv_path}'...",
#             flush=True,
#         )
#         df = pd.read_csv(csv_path)

#         for column in required_columns:
#             if column not in df.columns:
#                 df[column] = (
#                     False
#                     if column in {"is_full_dimension", "is_minimal"}
#                     else None
#                 )

#         df = df.drop_duplicates(
#             subset=["h", "exponent", "architecture"],
#             keep="last",
#         )
#     else:
#         print(
#             "No existing database found. Starting fresh...",
#             flush=True,
#         )
#         df = pd.DataFrame(columns=required_columns)

#     for h in h_values:
#         print(
#             f"\n--- RANDOM SEARCH: h={h}, exponent={exponent} ---",
#             flush=True,
#         )

#         num_hidden = h - 1
#         degree = exponent ** num_hidden
#         ambient_dim = (
#             math.comb(degree + d_0 - 1, d_0 - 1) * d_h
#         )

#         print(
#             f"  Target Ambient Dimension: {ambient_dim}",
#             flush=True,
#         )

#         # ---------------------------------------------------------------------
#         # Extract all known evaluations for this depth and exponent.
#         # ---------------------------------------------------------------------
#         evaluation_status: dict[tuple[int, ...], bool] = {}

#         if not df.empty:
#             subset_df = df[
#                 (df["h"] == h)
#                 & (df["exponent"] == exponent)
#             ]

#             for _, row in subset_df.iterrows():
#                 architecture = _parse_architecture(row["architecture"])

#                 if (
#                     len(architecture) != h + 1
#                     or architecture[0] != d_0
#                     or architecture[-1] != d_h
#                 ):
#                     continue

#                 hidden = architecture[1:-1]
#                 evaluation_status[hidden] = _as_bool(
#                     row["is_full_dimension"]
#                 )

#         known_minimal = _minimal_antichain(
#             hidden
#             for hidden, is_full in evaluation_status.items()
#             if is_full
#         )
#         known_short = _maximal_antichain(
#             hidden
#             for hidden, is_full in evaluation_status.items()
#             if not is_full
#         )

#         print(
#             f"  Loaded {len(evaluation_status):,} prior evaluation(s).",
#             flush=True,
#         )
#         print(
#             f"  Known filling boundary:    "
#             f"{sorted(known_minimal)}",
#             flush=True,
#         )
#         print(
#             f"  Known nonfilling boundary: "
#             f"{sorted(known_short)}",
#             flush=True,
#         )

#         # ---------------------------------------------------------------------
#         # Evaluate user guesses first, exactly as in the original workflow.
#         # ---------------------------------------------------------------------
#         if user_guesses and h in user_guesses:
#             for raw_guess in user_guesses[h]:
#                 guess = tuple(int(x) for x in raw_guess)

#                 if len(guess) != num_hidden:
#                     print(
#                         f"  [WARNING] Guess {guess} has length {len(guess)}, "
#                         f"but h={h} requires {num_hidden}. Skipping it.",
#                         flush=True,
#                     )
#                     continue

#                 if guess in evaluation_status:
#                     print(
#                         f"  Guess {guess} already appears in the CSV: "
#                         f"{'FILLING' if evaluation_status[guess] else 'SHORT'}.",
#                         flush=True,
#                     )
#                     continue

#                 print(
#                     f"  Evaluating Guess: {guess}...",
#                     flush=True,
#                 )

#                 result = evaluate_single_architecture(
#                     guess,
#                     h,
#                     d_0,
#                     d_h,
#                     exponent,
#                 )

#                 if not result:
#                     print(
#                         f"  [WARNING] No result returned for guess {guess}.",
#                         flush=True,
#                     )
#                     continue

#                 is_full = _as_bool(result["is_full_dimension"])
#                 evaluation_status[guess] = is_full

#                 df = _append_result(
#                     df,
#                     result,
#                     hidden=guess,
#                     h=h,
#                     exponent=exponent,
#                     d_0=d_0,
#                     d_h=d_h,
#                     ambient_dim=ambient_dim,
#                 )
#                 _save_csv(df, csv_path)

#                 print(
#                     f"    Dimension computed: "
#                     f"{result.get('dimension_computed', 'unknown')} / "
#                     f"{ambient_dim}",
#                     flush=True,
#                 )

#                 if is_full:
#                     known_minimal = _insert_minimal(
#                         known_minimal,
#                         guess,
#                     )
#                     print(
#                         f"    [FILLING] {guess}",
#                         flush=True,
#                     )
#                 else:
#                     known_short = _insert_maximal(
#                         known_short,
#                         guess,
#                     )
#                     print(
#                         f"    [SHORT] {guess}",
#                         flush=True,
#                     )

#         # ---------------------------------------------------------------------
#         # Determine exactly the same search box as the original code.
#         # ---------------------------------------------------------------------
#         print(
#             "  Generating randomized lazy frontier...",
#             flush=True,
#         )

#         if layer_bounds and h in layer_bounds:
#             bounds = layer_bounds[h]

#             if len(bounds) != num_hidden:
#                 print(
#                     f"  [WARNING] layer_bounds for h={h} has length "
#                     f"{len(bounds)}, but expected {num_hidden} hidden "
#                     "layers. Skipping this depth...",
#                     flush=True,
#                 )
#                 continue

#             lower = tuple(
#                 max(1, int(b_min))
#                 for b_min, _ in bounds
#             )
#             upper = tuple(
#                 int(b_max)
#                 for _, b_max in bounds
#             )

#             print(
#                 f"  Using custom per-layer bounds: {bounds}",
#                 flush=True,
#             )
#         else:
#             lower_value = max(1, int(min_width))
#             upper_value = int(max_width)

#             lower = (lower_value,) * num_hidden
#             upper = (upper_value,) * num_hidden

#             print(
#                 f"  Using global bounds: min_width={min_width}, "
#                 f"max_width={max_width}",
#                 flush=True,
#             )

#         if any(lo > hi for lo, hi in zip(lower, upper)):
#             print(
#                 f"  [WARNING] Inconsistent bounds: "
#                 f"lower={lower}, upper={upper}. "
#                 f"Skipping h={h}.",
#                 flush=True,
#             )
#             continue

#         formal_box_size = math.prod(
#             hi - lo + 1
#             for lo, hi in zip(lower, upper)
#         )

#         print(
#             f"  Formal box size: {formal_box_size:,}",
#             flush=True,
#         )
#         print(
#             "  The full candidate pool will not be constructed.",
#             flush=True,
#         )

#         # A fresh random generator preserves the original randomized behavior.
#         rng = py_random.Random()

#         def random_priority(
#             point: tuple[int, ...],
#         ) -> tuple[float, tuple[int, ...]]:
#             return rng.random(), point

#         heap: list[
#             tuple[
#                 tuple[float, tuple[int, ...]],
#                 tuple[int, ...],
#             ]
#         ] = [(random_priority(lower), lower)]

#         scheduled: set[tuple[int, ...]] = {lower}

#         def add_successors(
#             point: tuple[int, ...],
#         ) -> tuple[int, int, int]:
#             """
#             Generate immediate successors lazily.

#             Returns
#             -------
#             added:
#                 Number of successors inserted into the heap.
#             blocked_above_filling:
#                 Number rejected because they lie above a known filling point.
#             already_scheduled:
#                 Number already encountered earlier.
#             """
#             added = 0
#             blocked_above_filling = 0
#             already_scheduled = 0

#             for index in range(num_hidden):
#                 if point[index] >= upper[index]:
#                     continue

#                 successor_list = list(point)
#                 successor_list[index] += 1
#                 successor = tuple(successor_list)

#                 if successor in scheduled:
#                     already_scheduled += 1
#                     continue

#                 if any(
#                     _is_less_or_equal(filling_point, successor)
#                     for filling_point in known_minimal
#                 ):
#                     blocked_above_filling += 1
#                     scheduled.add(successor)
#                     continue

#                 scheduled.add(successor)
#                 heapq.heappush(
#                     heap,
#                     (random_priority(successor), successor),
#                 )
#                 added += 1

#             return (
#                 added,
#                 blocked_above_filling,
#                 already_scheduled,
#             )

#         def remove_queued_supersets(
#             filling_point: tuple[int, ...],
#         ) -> int:
#             """
#             Remove queued supersets of a newly found filling point.

#             This is safe because no such point can be a minimal filling
#             architecture, and none of its successors can be minimal either.
#             """
#             before = len(heap)

#             heap[:] = [
#                 item
#                 for item in heap
#                 if not _is_less_or_equal(
#                     filling_point,
#                     item[1],
#                 )
#             ]
#             heapq.heapify(heap)

#             return before - len(heap)

#         print(
#             f"  Starting randomized sequential evaluation with "
#             f"{len(heap)} exposed frontier point.",
#             flush=True,
#         )

#         # ---------------------------------------------------------------------
#         # Randomized lazy search.
#         # ---------------------------------------------------------------------
#         checked = 0
#         new_evaluations = 0
#         exact_csv_reuses = 0
#         parameter_prunes = 0
#         upward_prunes = 0
#         downward_prunes = 0

#         try:
#             while heap:
#                 _, target = heapq.heappop(heap)
#                 checked += 1

#                 print(
#                     f"\n  Checking {target} "
#                     f"(queue remaining: {len(heap):,})",
#                     flush=True,
#                 )

#                 # -------------------------------------------------------------
#                 # Upward pruning from known filling results.
#                 # -------------------------------------------------------------
#                 filling_witness = next(
#                     (
#                         filling_point
#                         for filling_point in known_minimal
#                         if _is_less_or_equal(
#                             filling_point,
#                             target,
#                         )
#                     ),
#                     None,
#                 )

#                 if filling_witness is not None:
#                     upward_prunes += 1
#                     print(
#                         f"    [Upward Pruned] {target} lies above known "
#                         f"filling architecture {filling_witness}.",
#                         flush=True,
#                     )
#                     continue

#                 # -------------------------------------------------------------
#                 # Downward pruning from known nonfilling results.
#                 #
#                 # In a lazy search, queued lower points are not physically
#                 # deleted, because their other successors may lead to
#                 # incomparable parts of the boundary. Instead, this point is
#                 # skipped and its immediate successors are exposed.
#                 # -------------------------------------------------------------
#                 short_witness = next(
#                     (
#                         short_point
#                         for short_point in known_short
#                         if _is_less_or_equal(
#                             target,
#                             short_point,
#                         )
#                     ),
#                     None,
#                 )

#                 if short_witness is not None:
#                     downward_prunes += 1
#                     added, blocked, duplicate = add_successors(target)

#                     print(
#                         f"    [Downward Pruned] {target} lies below known "
#                         f"nonfilling architecture {short_witness}.",
#                         flush=True,
#                     )
#                     print(
#                         f"    Exposed {added} successor(s); "
#                         f"{blocked} blocked above filling points; "
#                         f"{duplicate} already scheduled.",
#                         flush=True,
#                     )
#                     continue

#                 # -------------------------------------------------------------
#                 # Original cheap parameter-count pruning.
#                 # -------------------------------------------------------------
#                 params = _parameter_count(
#                     target,
#                     d_0,
#                     d_h,
#                 )

#                 print(
#                     f"    Parameter-count bound: {params}; "
#                     f"ambient target: {ambient_dim}.",
#                     flush=True,
#                 )

#                 if params < ambient_dim:
#                     parameter_prunes += 1
#                     known_short = _insert_maximal(
#                         known_short,
#                         target,
#                     )

#                     added, blocked, duplicate = add_successors(target)

#                     print(
#                         f"    [Dimension Pruned] {params} < "
#                         f"{ambient_dim}; no architecture evaluation needed.",
#                         flush=True,
#                     )
#                     print(
#                         f"    [Downward Pruned] The lower orthant of "
#                         f"{target} is now known nonfilling.",
#                         flush=True,
#                     )
#                     print(
#                         f"    Exposed {added} successor(s); "
#                         f"{blocked} blocked above filling points; "
#                         f"{duplicate} already scheduled.",
#                         flush=True,
#                     )
#                     continue

#                 # -------------------------------------------------------------
#                 # Reuse an exact result already present in the CSV.
#                 # -------------------------------------------------------------
#                 if target in evaluation_status:
#                     is_full = evaluation_status[target]
#                     exact_csv_reuses += 1

#                     print(
#                         f"    Reusing CSV result: "
#                         f"{'FILLING' if is_full else 'SHORT'}.",
#                         flush=True,
#                     )
#                 else:
#                     print(
#                         f"    Evaluating architecture "
#                         f"{[d_0, *target, d_h]}...",
#                         flush=True,
#                     )

#                     result = evaluate_single_architecture(
#                         target,
#                         h,
#                         d_0,
#                         d_h,
#                         exponent,
#                     )

#                     if not result:
#                         print(
#                             f"    [WARNING] No result returned for {target}. "
#                             "This branch is left unresolved for a future run.",
#                             flush=True,
#                         )
#                         continue

#                     is_full = _as_bool(
#                         result["is_full_dimension"]
#                     )
#                     evaluation_status[target] = is_full
#                     new_evaluations += 1

#                     print(
#                         f"    Dimension computed: "
#                         f"{result.get('dimension_computed', 'unknown')} / "
#                         f"{ambient_dim}.",
#                         flush=True,
#                     )

#                     df = _append_result(
#                         df,
#                         result,
#                         hidden=target,
#                         h=h,
#                         exponent=exponent,
#                         d_0=d_0,
#                         d_h=d_h,
#                         ambient_dim=ambient_dim,
#                     )

#                     # Save each completed expensive evaluation immediately.
#                     _save_csv(df, csv_path)

#                 # -------------------------------------------------------------
#                 # Update the monotone boundaries.
#                 # -------------------------------------------------------------
#                 if is_full:
#                     known_minimal = _insert_minimal(
#                         known_minimal,
#                         target,
#                     )

#                     removed = remove_queued_supersets(target)

#                     print(
#                         f"    [FILLING] {target}",
#                         flush=True,
#                     )
#                     print(
#                         f"    [Upward Pruned] {removed} queued superset(s) "
#                         "removed. No successors generated.",
#                         flush=True,
#                     )
#                 else:
#                     known_short = _insert_maximal(
#                         known_short,
#                         target,
#                     )

#                     added, blocked, duplicate = add_successors(target)

#                     print(
#                         f"    [SHORT] {target}",
#                         flush=True,
#                     )
#                     print(
#                         f"    [Downward Pruned] The lower orthant of "
#                         f"{target} is now known nonfilling.",
#                         flush=True,
#                     )
#                     print(
#                         f"    Exposed {added} successor(s); "
#                         f"{blocked} blocked above filling points; "
#                         f"{duplicate} already scheduled.",
#                         flush=True,
#                     )

#                 print(
#                     f"    Frontier status: queue={len(heap):,}, "
#                     f"scheduled={len(scheduled):,}, "
#                     f"filling boundary={len(known_minimal):,}, "
#                     f"nonfilling boundary={len(known_short):,}.",
#                     flush=True,
#                 )

#         except KeyboardInterrupt:
#             print(
#                 "\n[Interrupt] User halted the loop early. "
#                 "Saving completed evaluations to the CSV...",
#                 flush=True,
#             )

#             df = _recompute_minimality(df)
#             _save_csv(df, csv_path)

#             print(
#                 f"Database successfully saved to '{csv_path}'.",
#                 flush=True,
#             )
#             print(
#                 "Running the same cell again restarts the randomized "
#                 "frontier and reuses all existing CSV results for pruning.",
#                 flush=True,
#             )
#             return df

#         # ---------------------------------------------------------------------
#         # Finish this depth and save.
#         # ---------------------------------------------------------------------
#         print(
#             f"\n  Search complete for h={h}.",
#             flush=True,
#         )
#         print(
#             f"  Checked frontier points:      {checked:,}",
#             flush=True,
#         )
#         print(
#             f"  New architecture evaluations: {new_evaluations:,}",
#             flush=True,
#         )
#         print(
#             f"  Exact CSV result reuses:       {exact_csv_reuses:,}",
#             flush=True,
#         )
#         print(
#             f"  Parameter-count prunes:        {parameter_prunes:,}",
#             flush=True,
#         )
#         print(
#             f"  Upward order prunes:           {upward_prunes:,}",
#             flush=True,
#         )
#         print(
#             f"  Downward order prunes:         {downward_prunes:,}",
#             flush=True,
#         )
#         print(
#             f"  Current filling boundary:      "
#             f"{sorted(known_minimal)}",
#             flush=True,
#         )

#         print(
#             "Re-evaluating minimal filling properties "
#             "(grouped by depth h and exponent)...",
#             flush=True,
#         )

#         df = _recompute_minimality(df)
#         _save_csv(df, csv_path)

#         print(
#             f"Database successfully saved to '{csv_path}'.",
#             flush=True,
#         )

#     print(
#         f"\nAll requested searches completed. "
#         f"Database saved to '{csv_path}'.",
#         flush=True,
#     )

#     return df

# Execute Search

In [ ]:
# # This adjusts which depths you want to consider
# h_values_to_test = [2,3] 


# d0 = 2
# dh = 2
# r = 5

# # Optional: provide guess as tuple for the hidden layers
# my_guesses = {} 


# # Provide individual bounds for each hidden layer (length must match h - 1)
# # Format: { h_value: [(min1, max1), (min2, max2), ...] }
# custom_bounds = {
#     6: [
#         (3, 4),  # d1
#         (3, 10),  # d2
#         (5, 28),  # d3
#         (9, 82), # d4
#         (6, 18)  # d5
#     ],
# }

# # Run the bounded frontier search

# df_results = parameter_boundary_search(
#     h_values=h_values_to_test, 
#     max_width=300,        # Fallback if `h` not in layer_bounds
#     min_width=2,         # Fallback if `h` not in layer_bounds
#     exponent=r,          
#     d_0=d0,              
#     d_h=dh,              
#     csv_filename=f"../data/raw/{d0}_{dh}_r{r}_architectures.csv",
#     user_guesses=my_guesses,
#     layer_bounds=custom_bounds  # Injecting the new feature here
# )

No existing database found. Starting fresh...

--- RANDOM SEARCH: h=2, exponent=5 ---
  Target Ambient Dimension: 12
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 1 impossible/redundant architectures.
  Starting sequential evaluation on 298 viable candidates.
  [FULL ] [2, 239, 2] -> Rank: 12/12 (Params: 956)
    [Upward Pruned] 61 supersets removed from queue.
  [FULL ] [2, 218, 2] -> Rank: 12/12 (Params: 872)
    [Upward Pruned] 20 supersets removed from queue.
  [FULL ] [2, 198, 2] -> Rank: 12/12 (Params: 792)
    [Upward Pruned] 19 supersets removed from queue.


  [FULL ] [2, 111, 2] -> Rank: 12/12 (Params: 444)
    [Upward Pruned] 86 supersets removed from queue.
  [FULL ] [2, 84, 2] -> Rank: 12/12 (Params: 336)
    [Upward Pruned] 26 supersets removed from queue.
  [FULL ] [2, 26, 2] -> Rank: 12/12 (Params: 104)
    [Upward Pruned] 57 supersets removed from queue.
  [FULL ] [2, 9, 2] -> Rank: 12/12 (Params: 36)
    [Upward Pruned] 16 supersets removed from queue.
  [FULL ] [2, 6, 2] -> Rank: 12/12 (Params: 24)
    [Upward Pruned] 2 supersets removed from queue.
  [FULL ] [2, 4, 2] -> Rank: 12/12 (Params: 16)
    [Upward Pruned] 1 supersets removed from queue.
  [SHORT] [2, 3, 2] -> Rank: 9/12 (Params: 12)

--- RANDOM SEARCH: h=3, exponent=5 ---
  Target Ambient Dimension: 52
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 40 impossible/redundant architectures.
  Starting sequential evaluation on 89361 viable candidates.
  [FULL ] [2, 63, 81, 2] -> Rank: 52/52 (Params: 5391)
    [Upward Pruned] 52359 supersets removed from queue.
  [SHORT] [2, 3, 26, 2] -> Rank: 43/52 (Params: 136)
    [Downward Pruned] 31 subsets removed from queue.
  [FULL ] [2, 160, 39, 2] -> Rank: 52/52 (Params: 6638)
    [Upward Pruned] 5921 supersets removed from queue.
  [FULL ] [2, 56, 101, 2] -> Rank: 52/52 (Params: 5970)
    [Upward Pruned] 1399 supersets removed from queue.
  [FULL ] [2, 42, 58, 2] -> Rank: 52/52 (Params: 2636)
    [Upward Pruned] 5933 supersets removed from queue.
  [FULL ] [2, 18, 34, 2] -> Rank: 52/52 (Params: 716)
    [Upward Pruned] 9944 supersets removed from queue.
  [SHORT] [2, 273, 2, 2] -> Rank: 14/52 (Params: 1096)
    [Downward Pruned] 261 subsets removed from queue.
  [FULL ] [2, 171, 16, 2] -> Rank: 52/52 (Params: 3110)
    [Upward Pruned] 2339 supersets removed from queue.
  [FULL ] [2,

In [ ]:
# Looped version for searches


d0 = 2
dh = 2
r = 5

for d0 in range(1,10):
    for dh in range(1,10):
        for r in range(1,10):
            h_values_to_test = [2,3] 

            # Format: { h_value: [(min1, max1), (min2, max2), ...] }
            custom_bounds = {
                6: [
                    (3, 4),  # d1
                    (3, 10),  # d2
                    (5, 28),  # d3
                    (9, 82), # d4
                    (6, 18)  # d5
                ],
            }

            df_results = parameter_boundary_search(
                h_values=h_values_to_test, 
                max_width=300,        # Fallback if `h` not in layer_bounds
                min_width=2,         # Fallback if `h` not in layer_bounds
                exponent=r,          
                d_0=d0,              
                d_h=dh,              
                csv_filename=f"../data/raw/{d0}_{dh}_r{r}_architectures.csv",
                user_guesses=my_guesses,
                layer_bounds=custom_bounds  # Injecting the new feature here
            )

No existing database found. Starting fresh...

--- RANDOM SEARCH: h=2, exponent=1 ---
  Target Ambient Dimension: 1
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 0 impossible/redundant architectures.
  Starting sequential evaluation on 299 viable candidates.
  [FULL ] [1, 181, 1] -> Rank: 1/1 (Params: 362)
    [Upward Pruned] 119 supersets removed from queue.
  [FULL ] [1, 88, 1] -> Rank: 1/1 (Params: 176)
    [Upward Pruned] 92 supersets removed from queue.
  [FULL ] [1, 43, 1] -> Rank: 1/1 (Params: 86)
    [Upward Pruned] 44 supersets removed from queue.
  [FULL ] [1, 28, 1] -> Rank: 1/1 (Params: 56)
    [Upward Pruned] 14 supersets removed from queue.
  [FULL ] [1, 14, 1] -> Rank: 1/1 (Params: 28)
    [Upward Pruned] 13 supersets removed from queue.


  [FULL ] [1, 8, 1] -> Rank: 1/1 (Params: 16)
    [Upward Pruned] 5 supersets removed from queue.
  [FULL ] [1, 6, 1] -> Rank: 1/1 (Params: 12)
    [Upward Pruned] 1 supersets removed from queue.
  [FULL ] [1, 5, 1] -> Rank: 1/1 (Params: 10)
  [FULL ] [1, 2, 1] -> Rank: 1/1 (Params: 4)
    [Upward Pruned] 2 supersets removed from queue.

--- RANDOM SEARCH: h=3, exponent=1 ---
  Target Ambient Dimension: 1
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 0 impossible/redundant architectures.
  Starting sequential evaluation on 89401 viable candidates.
  [FULL ] [1, 252, 69, 1] -> Rank: 1/1 (Params: 17709)
    [Upward Pruned] 11367 supersets removed from queue.
  [FULL ] [1, 211, 275, 1] -> Rank: 1/1 (Params: 58511)
    [Upward Pruned] 1065 supersets removed from queue.
  [FULL ] [1, 218, 117, 1] -> Rank: 1/1 (Params: 25841)
    [Upward Pruned] 5371 supersets removed from queue.
  [FULL ] [1, 82, 259, 1] -> Rank: 1/1 (Params: 21579)
    [Upward Pruned] 5529 supersets removed from queue.
  [FULL ] [1, 46, 295, 1] -> Rank: 1/1 (Params: 13911)
    [Upward Pruned] 215 supersets removed from queue.
  [FULL ] [1, 32, 85, 1] -> Rank: 1/1 (Params: 2837)
    [Upward Pruned] 35335 supersets removed from queue.
  [FULL ] [1, 145, 62, 1] -> Rank: 1/1 (Params: 9197)
    [Upward Pruned] 2803 supersets removed from queue.
  [FULL ] [1, 142, 81, 1] -> Rank: 1/1 (Params: 11725)
    [Upward Pruned] 11 supersets removed from queue.
  [FULL ] [1, 

  Pruned 0 impossible/redundant architectures.
  Starting sequential evaluation on 299 viable candidates.
  [FULL ] [1, 162, 1] -> Rank: 1/1 (Params: 324)
    [Upward Pruned] 138 supersets removed from queue.
  [FULL ] [1, 85, 1] -> Rank: 1/1 (Params: 170)
    [Upward Pruned] 76 supersets removed from queue.
  [FULL ] [1, 40, 1] -> Rank: 1/1 (Params: 80)
    [Upward Pruned] 44 supersets removed from queue.
  [FULL ] [1, 32, 1] -> Rank: 1/1 (Params: 64)
    [Upward Pruned] 7 supersets removed from queue.
  [FULL ] [1, 27, 1] -> Rank: 1/1 (Params: 54)
    [Upward Pruned] 4 supersets removed from queue.


...

  Pruned 127 impossible/redundant architectures.
  Starting sequential evaluation on 89274 viable candidates.
  [FULL ] [2, 281, 157, 2] -> Rank: 100/100 (Params: 44993)
    [Upward Pruned] 2879 supersets removed from queue.
  [FULL ] [2, 244, 68, 2] -> Rank: 100/100 (Params: 17216)
    [Upward Pruned] 10400 supersets removed from queue.
  [FULL ] [2, 47, 161, 2] -> Rank: 100/100 (Params: 7983)
    [Upward Pruned] 27579 supersets removed from queue.
  [FULL ] [2, 102, 106, 2] -> Rank: 100/100 (Params: 11228)
    [Upward Pruned] 7809 supersets removed from queue.
  [FULL ] [2, 27, 256, 2] -> Rank: 100/100 (Params: 7478)
    [Upward Pruned] 899 supersets removed from queue.
  [FULL ] [2, 102, 12, 2] -> Rank: 100/100 (Params: 1452)
    [Upward Pruned] 16539 supersets removed from queue.
  [FULL ] [2, 39, 224, 2] -> Rank: 100/100 (Params: 9262)
    [Upward Pruned] 255 supersets removed from queue.
  [FULL ] [2, 27, 85, 2] -> Rank: 100/100 (Params: 2519)
    [Upward Pruned] 7343 supersets r

  Pruned 3 impossible/redundant architectures.
  Starting sequential evaluation on 296 viable candidates.
  [FULL ] [2, 25, 2] -> Rank: 18/18 (Params: 100)
    [Upward Pruned] 275 supersets removed from queue.
  [FULL ] [2, 7, 2] -> Rank: 18/18 (Params: 28)
    [Upward Pruned] 17 supersets removed from queue.
  [FULL ] [2, 6, 2] -> Rank: 18/18 (Params: 24)
  [SHORT] [2, 5, 2] -> Rank: 15/18 (Params: 20)

--- RANDOM SEARCH: h=3, exponent=8 ---
  Target Ambient Dimension: 130
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 198 impossible/redundant architectures.
  Starting sequential evaluation on 89203 viable candidates.
  [FULL ] [2, 94, 133, 2] -> Rank: 130/130 (Params: 12956)
    [Upward Pruned] 34775 supersets removed from queue.
  [FULL ] [2, 199, 61, 2] -> Rank: 130/130 (Params: 12659)
    [Upward Pruned] 7343 supersets removed from queue.
  [FULL ] [2, 279, 25, 2] -> Rank: 130/130 (Params: 7583)
    [Upward Pruned] 791 supersets removed from queue.
  [FULL ] [2, 209, 51, 2] -> Rank: 130/130 (Params: 11179)
    [Upward Pruned] 699 supersets removed from queue.
  [SHORT] [2, 143, 11, 2] -> Rank: 110/130 (Params: 1881)
    [Downward Pruned] 1273 subsets removed from queue.
  [FULL ] [2, 149, 37, 2] -> Rank: 130/130 (Params: 5885)
    [Upward Pruned] 6019 supersets removed from queue.
  [FULL ] [2, 37, 227, 2] -> Rank: 130/130 (Params: 8927)
    [Upward Pruned] 4217 supersets removed from queue.
  [FULL ] [2, 125, 131, 2] -> Rank: 130/130 (Params: 16887)
    [Upward Pruned] 47 supersets remo

  Pruned 3 impossible/redundant architectures.
  Starting sequential evaluation on 296 viable candidates.
  [FULL ] [2, 13, 2] -> Rank: 20/20 (Params: 52)
    [Upward Pruned] 287 supersets removed from queue.
  [FULL ] [2, 12, 2] -> Rank: 20/20 (Params: 48)
  [FULL ] [2, 8, 2] -> Rank: 20/20 (Params: 32)
    [Upward Pruned] 3 supersets removed from queue.
  [FULL ] [2, 7, 2] -> Rank: 20/20 (Params: 28)
  [SHORT] [2, 5, 2] -> Rank: 15/20 (Params: 20)
  [SHORT] [2, 6, 2] -> Rank: 18/20 (Params: 24)

--- RANDOM SEARCH: h=3, exponent=9 ---
  Target Ambient Dimension: 164
  Generating candidate pool...
  Using global bounds: min_width=2, max_width=300


  Pruned 277 impossible/redundant architectures.
  Starting sequential evaluation on 89124 viable candidates.
  [FULL ] [2, 189, 98, 2] -> Rank: 164/164 (Params: 19096)
    [Upward Pruned] 22735 supersets removed from queue.
  [FULL ] [2, 141, 79, 2] -> Rank: 164/164 (Params: 11579)
    [Upward Pruned] 12783 supersets removed from queue.
  [FULL ] [2, 93, 111, 2] -> Rank: 164/164 (Params: 10731)
    [Upward Pruned] 9119 supersets removed from queue.
  [FULL ] [2, 49, 44, 2] -> Rank: 164/164 (Params: 2342)
    [Upward Pruned] 20123 supersets removed from queue.
  [FULL ] [2, 35, 221, 2] -> Rank: 164/164 (Params: 8247)
    [Upward Pruned] 1119 supersets removed from queue.
  [FULL ] [2, 66, 43, 2] -> Rank: 164/164 (Params: 3056)
    [Upward Pruned] 234 supersets removed from queue.
  [FULL ] [2, 175, 17, 2] -> Rank: 164/164 (Params: 3359)
    [Upward Pruned] 3275 supersets removed from queue.
  [FULL ] [2, 20, 191, 2] -> Rank: 164/164 (Params: 4242)
    [Upward Pruned] 2069 supersets rem

In [180]:
# Display the minimal architectures found so far
print("\n=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===")
if not df_results.empty:
    minimal_archs = df_results[df_results['is_minimal'] == True]
    
    if not minimal_archs.empty:
        # Sort by parameters for easier reading
        minimal_archs = minimal_archs.sort_values(by="num_parameters")
        print(minimal_archs[["architecture", "num_parameters", "dimension_computed"]].to_string(index=False))
    else:
        print("No minimal full architectures found matching the criteria.")
else:
    print("Database is empty.")


=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===
 architecture  num_parameters  dimension_computed
    [2, 4, 2]              16                  12
 [2, 5, 8, 2]              66                  52
[2, 4, 10, 2]              68                  52


# Examining the Data Frame

In [181]:
# %load is_unimodal.py

import ast

def is_unimodal(data):
    # 1. Parse the string into a list safely
    if isinstance(data, str):
        try:
            # ast.literal_eval safely evaluates strings containing Python literals
            data = ast.literal_eval(data)
        except (ValueError, SyntaxError):
            raise ValueError(f"Could not parse the string: '{data}'. Ensure it is formatted like '[1, 2, 3]'.")
            
    # 2. Validate the data type
    if not isinstance(data, list):
        raise TypeError("Input must be a list or a string representation of a list.")
        
    # 3. Core unimodal logic
    n = len(data)
    if n <= 2:
        return True
        
    i = 0
    
    # Phase 1: Walk up the non-decreasing slope
    while i + 1 < n and data[i] <= data[i + 1]:
        i += 1
        
    # Phase 2: Walk down the non-increasing slope
    while i + 1 < n and data[i] >= data[i + 1]:
        i += 1
        
    # Phase 3: Check if we reached the end
    return i == n - 1

In [182]:
USER_DEPTH = 5
d0=3
dL=1
r=3

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)]))

------------------------------DISCOVERED FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
635,5,3,"[3, 21, 14, 55, 47, 1]",3759,3403,3403,True,False,False
637,5,3,"[3, 20, 47, 46, 50, 1]",5512,3403,3403,True,False,False
640,5,3,"[3, 11, 44, 55, 48, 1]",5625,3403,3403,True,False,True
641,5,3,"[3, 42, 41, 53, 30, 1]",5641,3403,3403,True,False,False
642,5,3,"[3, 21, 49, 55, 42, 1]",6139,3403,3403,True,False,True
...,...,...,...,...,...,...,...,...,...
12247,5,3,"[3, 6, 48, 53, 13, 1]",3552,3403,3403,True,True,True
12250,5,3,"[3, 8, 45, 33, 51, 1]",3603,3403,3403,True,True,False
12251,5,3,"[3, 8, 33, 47, 36, 1]",3567,3403,3403,True,True,True
12255,5,3,"[3, 6, 35, 46, 37, 1]",3577,3403,3403,True,True,True


5508
------------------------------DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
1345,5,3,"[3, 8, 37, 42, 39, 1]",3551,3403,3403,True,True,True
1442,5,3,"[3, 9, 37, 51, 25, 1]",3547,3403,3403,True,True,True
1625,5,3,"[3, 7, 43, 35, 48, 1]",3555,3403,3403,True,True,False
1784,5,3,"[3, 9, 38, 50, 25, 1]",3544,3403,3403,True,True,True
1962,5,3,"[3, 8, 36, 41, 42, 1]",3552,3403,3403,True,True,True
...,...,...,...,...,...,...,...,...,...
12247,5,3,"[3, 6, 48, 53, 13, 1]",3552,3403,3403,True,True,True
12250,5,3,"[3, 8, 45, 33, 51, 1]",3603,3403,3403,True,True,False
12251,5,3,"[3, 8, 33, 47, 36, 1]",3567,3403,3403,True,True,True
12255,5,3,"[3, 6, 35, 46, 37, 1]",3577,3403,3403,True,True,True


1630
------------------------------DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
1625,5,3,"[3, 7, 43, 35, 48, 1]",3555,3403,3403,True,True,False
2575,5,3,"[3, 7, 45, 37, 42, 1]",3597,3403,3403,True,True,False
2963,5,3,"[3, 6, 47, 35, 45, 1]",3565,3403,3403,True,True,False
3027,5,3,"[3, 6, 46, 35, 47, 1]",3596,3403,3403,True,True,False
3097,5,3,"[3, 15, 42, 36, 44, 1]",3815,3403,3403,True,True,False
...,...,...,...,...,...,...,...,...,...
12198,5,3,"[3, 13, 9, 54, 54, 1]",3612,3403,3403,True,True,False
12215,5,3,"[3, 8, 39, 37, 47, 1]",3565,3403,3403,True,True,False
12231,5,3,"[3, 10, 44, 36, 41, 1]",3571,3403,3403,True,True,False
12239,5,3,"[3, 13, 37, 34, 55, 1]",3703,3403,3403,True,True,False


195
